# Honeybee nest: masked-convolutional autoregressive model

Reproduces the paper's nest-content **prediction** pipeline end to end: load
and relabel the 4D colony grids, spatially align and crop them, define the
masked-convolutional `SpatialClassifier`, assemble the 18-hive evaluation set
(6 control + 6 physically shuffled + 6 simulated-shuffled colonies), run the
eight trained checkpoints, and regenerate the paper's model figures.

The trained weights (`spatial_classifier_v2_*.pth`, committed under `models/`)
are **used as-is** -- the training loop below is kept as a reference so the
checkpoints' provenance is visible, but the notebook does not re-train.


## 1. Load and relabel the colony grids

Load the combined 4D atlas and remap the 21 detailed content classes to the six
simplified classes the model is trained with.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

# Original class names (combined, detailed)
combined_names = [
    'background', 'wood', 'empty_comb', 'eggs', 'larvae', 'pupae', 'pollen', 'nectar',
    'honey', 'queen_cup', 'queen_cell', 'bees', 'drone_empty_comb', 'drone_eggs',
    'drone_larvae', 'drone_pupae', 'drone_pollen', 'drone_nectar', 'drone_honey', '?', '?'
]

# Simplified class names
simple_names = [
    'background',      # 0
    'wood',            # 1
    'empty_comb',      # 2
    'any_brood',       # 3
    'pollen',          # 4
    'nectar_honey',    # 5
]

# Colony IDs (still used if needed)
colony_names = ['DD1', 'DD2', 'DD4', 'DD5', 'DD6', 'DD7', 'SH2', 'SH3', 'SH4', 'SH5', 'SH6', 'SH7']

# Original loading and setup (commented out)
# hive_grids = np.load("AllColoniesCombinedDS48.npy", allow_pickle=True)
# NUM_CLASSES = len(combined_names)
# hive_grids = np.load("ContentsSimple.npy", allow_pickle=True)

# Load detailed labels file
# hive_grids = np.load("AllColoniesCombinedDS48_DD_SH.npy", allow_pickle=True)
hive_grids = np.load("AllColoniesCombinedDS48v2.npy", allow_pickle=True)

# === Apply label remapping ===

# Map from combined index → simplified index
label_map = {
    0: 0,   # background → background
    1: 1,   # wood → wood
    2: 2,   # empty_comb → empty_comb
    3: 3,   # eggs → any_brood
    4: 3,   # larvae → any_brood
    5: 3,   # pupae → any_brood
    6: 4,   # pollen → pollen
    7: 5,   # nectar → nectar_honey
    8: 5,   # honey → nectar_honey
    9: 2,   # queen_cup → empty_comb ✅ (empty cell)
    10: 3,  # queen_cell → any_brood ✅ (contains developing queen)
    11: 0,  # bees → background ✅
    12: 2,  # drone_empty_comb → empty_comb
    13: 3,  # drone_eggs → any_brood
    14: 3,  # drone_larvae → any_brood
    15: 3,  # drone_pupae → any_brood
    16: 4,  # drone_pollen → pollen
    17: 5,  # drone_nectar → nectar_honey
    18: 5,  # drone_honey → nectar_honey
    19: 0,  # ? → background
    20: 0,  # ? → background
}

default_label = 0  # Fallback: background

def map_labels(grid):
    return np.vectorize(lambda x: label_map.get(x, default_label))(grid)

# Map every hive grid in the list
hive_grids = [map_labels(hive) for hive in hive_grids]

# Set final class list and number of classes
combined_names = simple_names
NUM_CLASSES = len(simple_names)

## 2. Spatial alignment and cropping

Each colony is (weeks, 20, H, W) where the 20 "frame sides" alternate Side A /
Side B of the 10 frames. Frame faces are aligned to a global mean-wood mask by
phase cross-correlation (integer pixel shifts), then all colonies are cropped to
the largest wood contour.

In [2]:
import numpy as np
from skimage.registration import phase_cross_correlation

def align_frame_skimage_with_shift(reference_frame, moving_frame, original_frame, grid_index=None, frame_indices=None):
    """
    Compute alignment shifts using binary frames (reference and moving),
    but apply the computed shift to the original frame.
    """
    # Check for zero variance
    if np.var(reference_frame) == 0 or np.var(moving_frame) == 0:
        if grid_index is not None and frame_indices is not None:
            print(f"Zero variance detected in grid {grid_index}, frame indices {frame_indices}")
        return np.zeros_like(original_frame)  # Return a blank frame if inputs are invalid

    # Compute the offset using phase cross-correlation
    try:
        shift, error, diffphase = phase_cross_correlation(reference_frame, moving_frame, upsample_factor=1)
    except Exception as e:
        if grid_index is not None and frame_indices is not None:
            print(f"Error during cross-correlation in grid {grid_index}, frame indices {frame_indices}: {e}")
        return np.zeros_like(original_frame)  # Return a blank frame if computation fails

    # Check for invalid error values
    if np.isnan(error) or np.isinf(error):
        if grid_index is not None and frame_indices is not None:
            print(f"Invalid error value detected in grid {grid_index}, frame indices {frame_indices}")
        return np.zeros_like(original_frame)

    # Round offsets to integer values
    shift = np.round(shift).astype(int)
    x_shift, y_shift = shift

    # Initialize the aligned frame with zeros
    aligned_frame = np.zeros_like(original_frame)

    # Apply the computed shift to the original frame
    if x_shift > 0:  # Shift down
        aligned_frame[x_shift:, :] = original_frame[:-x_shift, :]
    elif x_shift < 0:  # Shift up
        aligned_frame[:x_shift, :] = original_frame[-x_shift:, :]
    else:  # No vertical shift
        aligned_frame[:, :] = original_frame[:, :]

    if y_shift > 0:  # Shift right
        aligned_frame[:, y_shift:] = aligned_frame[:, :-y_shift]
    elif y_shift < 0:  # Shift left
        aligned_frame[:, :y_shift] = aligned_frame[:, -y_shift:]

    return aligned_frame


# Step 1: Compute the global mean wood class (class 1) across all hive grids (Side A only)
side1_frames = []
for hive_data in hive_grids:
    side1_grid = hive_data[:, ::2, :, :]  # Extract Side A
    side1_frames.append(side1_grid)

side1_frames_combined = np.concatenate(
    [grid.reshape(-1, grid.shape[-2], grid.shape[-1]) for grid in side1_frames], axis=0
)
mean_wood_class_1 = np.mean((side1_frames_combined == 1).astype(np.float32), axis=0)  # Global mean


# Step 2: Align both sides to the global mean and then align Side B to Side A
hive_grids_aligned = []

for grid_index, hive_data in enumerate(hive_grids):  # Add grid_index to loop
    # Split into sides
    side1_grid = hive_data[:, ::2, :, :]  # Side A
    side2_grid = hive_data[:, 1::2, :, :]  # Side B
    mirrored_side2_grid = np.flip(side2_grid, axis=-1)

    # Align both sides to the global mean wood class
    aligned_side1_grid = np.zeros_like(side1_grid)
    aligned_side2_grid = np.zeros_like(mirrored_side2_grid)

    for i in range(side1_grid.shape[0]):  # Iterate over weeks
        for j in range(side1_grid.shape[1]):  # Iterate over frames
            # Align Side A to mean wood class
            aligned_side1_grid[i, j, :, :] = align_frame_skimage_with_shift(
                mean_wood_class_1,  # Reference: global mean wood class 1
                (side1_grid[i, j, :, :] == 1).astype(np.float32),  # Binary wood class 1 frame
                side1_grid[i, j, :, :],  # Original frame to apply the shift
                grid_index=grid_index,  # Debug info
                frame_indices=(i, j)
            )

            # Align mirrored Side B to mean wood class
            aligned_side2_grid[i, j, :, :] = align_frame_skimage_with_shift(
                mean_wood_class_1,  # Reference: global mean wood class 1
                (mirrored_side2_grid[i, j, :, :] == 1).astype(np.float32),  # Binary wood class 1 frame
                mirrored_side2_grid[i, j, :, :],  # Original frame to apply the shift
                grid_index=grid_index,  # Debug info
                frame_indices=(i, j)
            )

    # Align Side B to Side A using wood and comb classes (1 and 2)
    final_aligned_side2_grid = np.zeros_like(aligned_side2_grid)

    for i in range(aligned_side1_grid.shape[0]):  # Iterate over weeks
        for j in range(aligned_side1_grid.shape[1]):  # Iterate over frames
            final_aligned_side2_grid[i, j, :, :] = align_frame_skimage_with_shift(
                # ((aligned_side1_grid[i, j, :, :] >= 1) & (aligned_side1_grid[i, j, :, :] != 11)).astype(np.float32),
                # ((aligned_side2_grid[i, j, :, :] >= 1) & (aligned_side2_grid[i, j, :, :] != 11)).astype(np.float32),
                (aligned_side1_grid[i, j, :, :] >= 1).astype(np.float32),
                (aligned_side2_grid[i, j, :, :] >= 1).astype(np.float32),
                aligned_side2_grid[i, j, :, :],  # Original frame of Side B to apply the shift
                grid_index=grid_index,  # Debug info
                frame_indices=(i, j)
            )

    # Recombine sides into a single grid
    combined_grid = np.zeros_like(hive_data)
    combined_grid[:, ::2, :, :] = aligned_side1_grid
    combined_grid[:, 1::2, :, :] = final_aligned_side2_grid

    # Append the aligned grid
    hive_grids_aligned.append(combined_grid)


Zero variance detected in grid 1, frame indices (0, 2)
Zero variance detected in grid 1, frame indices (0, 2)
Zero variance detected in grid 4, frame indices (0, 9)
Zero variance detected in grid 4, frame indices (0, 9)
Zero variance detected in grid 9, frame indices (0, 3)
Zero variance detected in grid 9, frame indices (0, 3)


In [3]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import measure

# ---- tweakables ----
MARGIN = 1     # pixels of padding on each side
THRESH  = 0.5  # threshold for contouring on the mean mask

# 1) Build mean wood mask from aligned hive grids (==1 is wood)
# hive_grids_aligned: iterable/list of arrays with shape (G1, G2, H, W)
H, W = np.asarray(hive_grids_aligned[0]).shape[-2:]
all_frames = np.concatenate(
    [np.asarray(grid).reshape(-1, H, W) for grid in hive_grids_aligned],
    axis=0
)  # shape: (total_frames, H, W)

wood_class_1_mask = (all_frames == 1).astype(np.float32)   # (F, H, W)
mean_mask = wood_class_1_mask.mean(axis=0)                 # (H, W) in [0,1]

# 2) Find biggest contour on the mean mask at level=THRESH
contours = measure.find_contours(mean_mask, level=THRESH)
if not contours:
    raise RuntimeError("No contour found on the mean mask — adjust THRESH.")

def poly_area(c):
    y, x = c[:, 0], c[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))

largest = max(contours, key=poly_area)

# 3) Bbox (rows, cols), add integer margin and clip to image
rmin, rmax = largest[:, 0].min(), largest[:, 0].max()
cmin, cmax = largest[:, 1].min(), largest[:, 1].max()
rmin, rmax = int(np.floor(rmin)), int(np.ceil(rmax))
cmin, cmax = int(np.floor(cmin)), int(np.ceil(cmax))

rmin = max(0, rmin - MARGIN)
rmax = min(H - 1, rmax + MARGIN)
cmin = max(0, cmin - MARGIN)
cmax = min(W - 1, cmax + MARGIN)
bbox = (rmin, rmax, cmin, cmax)


# 5) Crop every aligned grid on the last two dims (…, H, W)
def crop_last2(a, bbox):
    rmin, rmax, cmin, cmax = bbox
    a = np.asarray(a)
    sl = (slice(None),) * (a.ndim - 2) + (slice(rmin, rmax + 1), slice(cmin, cmax + 1))
    return a[sl]

cropped_hive_grids_aligned = [crop_last2(g, bbox) for g in hive_grids_aligned]

print("bbox (inclusive):", bbox)
print("original shape:", np.asarray(hive_grids_aligned[0]).shape)
print("cropped  shape:", np.asarray(cropped_hive_grids_aligned[0]).shape)
# (crop bbox visualization omitted in the public notebook)

bbox (inclusive): (13, 64, 7, 110)
original shape: (20, 20, 77, 116)
cropped  shape: (20, 20, 52, 104)


In [4]:
# augmented_hive_grids = []
# # Augment hive_grids_side2
# for grid in cropped_hive_grids_aligned:
#     # Original
#     augmented_hive_grids.append(grid)
#     # Flip along the first axis
#     augmented_hive_grids.append(grid[:, ::-1, :, :])
#     # Flip along the last axis
#     augmented_hive_grids.append(grid[:, :, :, ::-1])
#     # Flip along both axes
#     augmented_hive_grids.append(grid[:, ::-1, :, ::-1])



In [5]:
# # Combine the augmented grids back into a single list `hive_grids`
# hive_grids = augmented_hive_grids
hive_grids = cropped_hive_grids_aligned

# Now `hive_grids` contains the augmented versions of both sides.
hive_grids = [grid.copy() for grid in hive_grids]

In [6]:
for grid in hive_grids:
    print(grid.shape)

(20, 20, 52, 104)
(20, 20, 52, 104)
(20, 20, 52, 104)
(20, 20, 52, 104)
(20, 20, 52, 104)
(17, 20, 52, 104)
(20, 20, 52, 104)
(20, 20, 52, 104)
(14, 20, 52, 104)
(20, 20, 52, 104)
(20, 20, 52, 104)
(17, 20, 52, 104)


In [7]:
import numpy as np

# Example: hive_grids is a list of numpy arrays
# hive_grids = [np.array([...]), np.array([...]), ...]
for grid in hive_grids:
    grid[grid == 255] = 0

# Get all unique class labels across all arrays
unique_labels = np.unique(np.concatenate(hive_grids))

# Create a mapping from old labels to consecutive integers in ascending order
label_mapping = {old_label: new_label for new_label, old_label in enumerate(sorted(unique_labels))}

# Relabel the class labels in each array to make them consecutive integers
hive_grids = [np.vectorize(label_mapping.get)(grid) for grid in hive_grids]

# Verify that the labels are now consecutive integers
new_unique_labels = np.unique(np.concatenate(hive_grids))
print("New unique labels:", new_unique_labels)


New unique labels: [0 1 2 3 4 5]


## 3. Model

`MaskedConv3d` (a `Conv3d` whose center tap is zeroed and whose weights use
masked LeCun-normal init), `MultilinearConv3d`, and the autoregressive
`SpatialClassifier`: class and positional parameters plus an optional masked
contextual convolution over the spatial axes.

In [8]:
import torch
import math
from torch.nn import init
from einops import rearrange

import torch
import torch.nn as nn
import math

def lecun_normal_masked_conv3d(tensor, mask_center=True):
    """
    Custom LeCun Normal Initialization for MaskedConv3d kernels.
    Adjusts for the masked center pixel.
    
    Args:
        tensor (torch.Tensor): The weight tensor to initialize (out_channels, in_channels, depth, height, width).
        mask_center (bool): If True, adjusts the variance to exclude the masked center pixel.
    """
    if not isinstance(tensor, torch.Tensor):
        raise ValueError("The tensor to initialize must be a PyTorch Tensor.")

    # Calculate the variance
    fan_in = tensor.size(1) * tensor.size(2) * tensor.size(3) * tensor.size(4)  # Total input connections per filter
    if mask_center:
        fan_in -= 1  # Exclude the center pixel
    
    # Compute standard deviation
    std = math.sqrt(1.0 / fan_in)
    
    # Initialize weights with normal distribution
    with torch.no_grad():
        tensor.normal_(0, std)

class MaskedConv3d(nn.Conv3d):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, dilation=1, groups=1, bias=True, padding_mode='zeros'):
        """
        Subclass of nn.Conv3d with masking applied to the center of the kernel and custom initialization.
        Args:
            in_channels (int): Number of input channels.
            out_channels (int): Number of output channels.
            kernel_size (int or tuple): Size of the convolving kernel.
            stride (int or tuple, optional): Stride of the convolution. Default: 1
            padding (int or tuple, optional): Zero-padding added to all three sides of the input. Default: 0
            dilation (int or tuple, optional): Spacing between kernel elements. Default: 1
            groups (int, optional): Number of blocked connections from input channels to output channels. Default: 1
            bias (bool, optional): If True, adds a learnable bias to the output. Default: True
            padding_mode (str, optional): 'zeros', 'reflect', 'replicate' or 'circular'. Default: 'zeros'
        """
        super(MaskedConv3d, self).__init__(
            in_channels, out_channels, kernel_size, stride, padding, dilation, groups, bias, padding_mode
        )

        # Apply custom LeCun normal initialization for weights
        lecun_normal_masked_conv3d(self.weight, mask_center=True)

        # Initialize biases to zero
        if self.bias is not None:
            nn.init.zeros_(self.bias)
    
    def forward(self, input):
        # Apply the center mask to the kernel before forward pass
        self.mask_center()
        return super(MaskedConv3d, self).forward(input)
    
    def mask_center(self):
        """
        Masks the center pixel of the kernel weights to zero.
        This prevents the center pixel from contributing to predictions.
        """
        with torch.no_grad():
            # Get the center coordinates of the kernel
            center_coords = tuple(dim // 2 for dim in self.weight.shape[2:])  # depth, height, width
            # Set the center of all kernels to zero
            self.weight[:, :, center_coords[0], center_coords[1], center_coords[2]] = 0.0

import math
import torch
import torch.nn as nn

class MultilinearConv3d(nn.Module):
    def __init__(self, embedding_dim, order):
        super(MultilinearConv3d, self).__init__()
        self.embedding_dim = embedding_dim
        self.order = max(1, order)  # Ensure order is at least 1

        # Number of chunks is order + 1 (e.g., order=1 means 2 chunks, order=2 means 3 chunks)
        self.num_chunks = self.order + 1

        # Define Conv3D pointwise layers (equivalent to linear layers in this case)
        self.conv = nn.Conv3d(
            in_channels=embedding_dim,  # Embedding dimension as input channels
            out_channels=self.num_chunks * embedding_dim,  # Output dimensions for all chunks
            kernel_size=1,  # Pointwise convolution
            bias=False
        )

        # Initialize weights using Lecun Normal and bias to zero
        nn.init.kaiming_normal_(self.conv.weight, mode='fan_in', nonlinearity='linear')

    def forward(self, x):
        # x is expected to be of shape (batch_size, embedding_dim, depth, height, width)

        # Apply Conv3D (pointwise equivalent to linear transformation)
        transformed = self.conv(x)  # Shape: (batch_size, num_chunks * embedding_dim, depth, height, width)

        # Reshape for processing interactions
        batch_size, channels, depth, height, width = transformed.size()
        transformed = transformed.view(batch_size, self.num_chunks, self.embedding_dim, depth, height, width)

        # Initialize output
        out = torch.zeros_like(x)  # Initialize output with zeros

        # Always start interaction with x0 * x1
        interaction = transformed[:, 0] * transformed[:, 1]
        out = out + interaction

        for idx in range(2, self.num_chunks):
            # Compute subsequent interactions
            interaction = interaction * transformed[:, idx]
            out = out + interaction

        # Residual connection
        return x + out  # Residual connection

import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange
import warnings

class SpatialClassifier(nn.Module):
    def __init__(self, /, *, num_classes, num_depth, num_height, num_width, embedding_dim=16, kernel_size=3, dilation=1,
                 depth=True, height=True, width=True, pos_embed=False, context=True, interaction=0):
        super(SpatialClassifier, self).__init__()

        kwargs = self.get_kwargs(
            num_classes=num_classes,
            embedding_dim=embedding_dim,
            kernel_size=kernel_size,
            dilation=dilation,
            depth=depth,
            height=height,
            width=width,
            pos_embed=pos_embed,
            context=context,
            num_depth=num_depth,
            num_height=num_height,
            num_width=num_width,
            interaction=interaction
        )

        # Assign processed kwargs to attributes
        self.num_classes = kwargs['num_classes']
        self.embedding_dim = kwargs['embedding_dim']
        self.rsqrt_dim = self.embedding_dim ** -0.5
        self.context = kwargs['context']
        self.pos_embed = kwargs['pos_embed']
        self.num_depth = kwargs['num_depth']
        self.num_height = kwargs['num_height']
        self.num_width = kwargs['num_width']
        self.interaction = kwargs['interaction']

        # Embedding weights and bias
        # self.embedding = FactorizedEmbedding(self.embedding_dim)
        self.embedding_weights = nn.Parameter(torch.randn(self.num_classes, self.embedding_dim) * self.rsqrt_dim)
        self.bias = nn.Parameter(torch.randn(self.embedding_dim) * self.rsqrt_dim)  # Simplified bias

        # Contextual convolution (only initialized if context=True)
        if self.context:
            spatial_dims = (
                kwargs['kernel_size'] if kwargs['depth'] else 1,
                kwargs['kernel_size'] if kwargs['height'] else 1,
                kwargs['kernel_size'] if kwargs['width'] else 1,
            )
            self.conv = MaskedConv3d(
                self.embedding_dim,
                self.embedding_dim,
                spatial_dims,
                padding="same",
                bias=False,
                dilation=kwargs['dilation']
            )
            self.multilinear = MultilinearConv3d(self.embedding_dim, self.interaction) if self.interaction >= 1 else None
        else:
            self.conv = None
            self.multilinear = None

        # Positional embedding parameters
        if self.pos_embed:
            self.positional_embedding = nn.Parameter(
                torch.randn(1, self.embedding_dim, self.num_depth, self.num_height, self.num_width) * self.rsqrt_dim
            )

    @staticmethod
    def get_kwargs(num_classes, embedding_dim, kernel_size, dilation, depth, height, width,
                   pos_embed, context, num_depth, num_height, num_width, interaction):
        """
        Processes input arguments, applies validation and warnings, and returns the normalized configuration dictionary.
        """
        # Validate context and kernel size
        if context and kernel_size <= 1:
            warnings.warn(
                f"`context` is set to True but `kernel_size` is {kernel_size}, which is <= 1. "
                "Setting `context` to False.",
                UserWarning
            )
            context = False

        # If pos_embed and context are both False, disable depth, height, and width
        if not (pos_embed or context):
            depth = height = width = False

        # Ensure kernel_size is 1 if context is False
        if not context:
            kernel_size = 1

        if not (depth or height or width):
            if pos_embed:
                warnings.warn(
                    "`pos_embed` is set to True but `depth`, `height`, and `width` are all False. "
                    "Setting `pos_embed` to False.",
                    UserWarning
                )
                pos_embed = False
            if context:
                warnings.warn(
                    "`context` is set to True but `depth`, `height`, and `width` are all False. "
                    "Setting `context` to False.",
                    UserWarning
                )
                context = False

        # Validate interaction
        if not isinstance(interaction, (int, float)) or interaction < 0:
            warnings.warn(
                f"`interaction` is set to {interaction}. Interaction must be a numeric value >= 0. "
                "Setting `interaction` to 0.",
                UserWarning
            )
            interaction = 0
        elif interaction >= 1 and not context:
            warnings.warn(
                f"`interaction` is set to {interaction} but `context` is False. "
                "Interaction requires `context` to be True. Setting `interaction` to 0.",
                UserWarning
            )
            interaction = 0

        normalized_kernel_size = kernel_size if any([depth, height, width]) else 1

        num_depth = num_depth if depth else 1
        num_height = num_height if height else 1
        num_width = num_width if width else 1

        return {
            'num_classes': num_classes,
            'embedding_dim': embedding_dim,
            'kernel_size': normalized_kernel_size,
            'dilation': dilation,
            'depth': depth,
            'height': height,
            'width': width,
            'pos_embed': pos_embed,
            'context': context,
            'num_depth': num_depth,
            'num_height': num_height,
            'num_width': num_width,
            'interaction': interaction
        }

    def forward(self, x):
        batch_size, depth_size, height_size, width_size = x.size()

        embedding_weights = self.embedding_weights + self.bias
        # embedding_weights = self.embedding.get_combined_embedding_weights() + self.bias
        
        if self.context:
            x = F.embedding(x, embedding_weights)
            x = rearrange(x, 'b d h w c -> b c d h w')  # Rearrange for convolution
            x = self.conv(x) + self.bias.view(1, -1, 1, 1, 1) 
            if self.pos_embed:
                x = x + self.positional_embedding
            if self.multilinear:
                x = self.multilinear(x)
            logits = torch.einsum('b c d h w, co -> b o d h w', x, embedding_weights.T)
        else:
            if self.pos_embed:
                x = self.positional_embedding + self.bias.view(1, -1, 1, 1, 1)
                logits = torch.einsum('b c d h w, co -> b o d h w', x, embedding_weights.T)
                logits = logits.expand(
                    batch_size, self.num_classes, depth_size, height_size, width_size
                )
            else:
                logits = torch.einsum('c, co -> o', self.bias, embedding_weights.T)
                logits = logits.view(1, self.num_classes, 1, 1, 1).expand(
                    batch_size, self.num_classes, depth_size, height_size, width_size
                )

        return logits


## 4. Dataset, metrics, and training

`GridDataset` yields 3D frames; `evaluate_spatial_classifier` returns the
per-hive ground-truth / predictions / entropy / log-likelihood the figures are
built from. `train_spatial_classifier` (PyTorch Lightning) is the reference
recipe for the committed checkpoints.

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader, Dataset
from pytorch_lightning import seed_everything
import pytorch_lightning as pl
from torch.utils.data import DataLoader, Dataset
import torch
import torch.nn as nn
import os

class GridDataset(Dataset):
    def __init__(self, grids):
        """
        Args:
            grids (list of numpy.ndarray or torch.Tensor): List of 3D grid data with varying dimensions.
        """
        # Ensure each grid is a torch tensor with consistent dtype
        self.grids = [torch.tensor(grid, dtype=torch.long) for grid in grids]
        # Precompute global-to-local index mapping
        self.grid_mapping = []
        for grid_idx, grid in enumerate(self.grids):
            dim0_size = grid.shape[0]
            self.grid_mapping.extend([(grid_idx, local_idx) for local_idx in range(dim0_size)])

        # Calculate number of unique classes
        self.num_classes = len(torch.unique(torch.cat([grid.flatten() for grid in self.grids])))
        self.shape = (len(self), *self.grids[0].shape[1:])

    def __len__(self):
        # Total number of entries across all grids
        return len(self.grid_mapping)
    
    def __getitem__(self, idx):
        # Use precomputed mapping to find the grid and local index
        hive_idx, week_idx = self.grid_mapping[idx]
        grid_sample = self.grids[hive_idx][week_idx]
        return grid_sample, hive_idx, week_idx

import torch

def compute_class_weights(dataset, normalize=True):
    """
    Computes class weights for a dataset containing grids of 3D data.
    
    Args:
        dataset (GridDataset): Instance of the GridDataset class.
        normalize (bool): Whether to normalize weights to sum to 1.
        
    Returns:
        class_weights (torch.Tensor): Tensor containing weights for each class.
    """
    # Flatten all grids to count class frequencies
    all_classes = torch.cat([grid.flatten() for grid in dataset.grids], dim=0)
    
    # Count occurrences of each class using bincount for efficiency
    class_counts = torch.bincount(all_classes)
    total_samples = all_classes.size(0)
    
    # Compute class weights (inverse frequency)
    class_weights = total_samples / (class_counts + 1e-6)  # Avoid division by zero
    
    if normalize:
        # Normalize weights to sum to 1
        class_weights /= class_weights.sum()
    
    return class_weights

dataset = GridDataset(hive_grids)

# Compute weights
class_weights = compute_class_weights(dataset)



import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from collections import defaultdict

def evaluate_spatial_classifier(model, hive_grids, device, batch_size=8):
    dataset = GridDataset(hive_grids)
    eval_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    model.eval()
    criterion = torch.nn.CrossEntropyLoss(reduction='none')

    hive_results = defaultdict(lambda: {
        'ground_truth': [],
        'predictions': [],
        'entropy': [],
        'log_likelihood': [],
    })

    with torch.no_grad():
        for grids, hive_idx, week_idx in eval_loader:
            # grids: [B, D, H, W]
            grids = grids.to(device, dtype=torch.long)
            logits = model(grids)  # Expected: [B, C, D, H, W]

            if logits.dim() == 5:
                logits = logits.permute(0, 2, 3, 4, 1)  # [B, D, H, W, C]
                B, D, H, W, C = logits.shape
            else:
                raise ValueError(f"Expected logits to be 5D [B, C, D, H, W], got shape {logits.shape}")

            logits_flat = logits.view(-1, C)
            targets_flat = grids.view(-1)

            preds = torch.argmax(logits_flat, dim=1)
            probs = F.softmax(logits_flat, dim=1)
            log_probs = F.log_softmax(logits_flat, dim=1)
            entropy = -(probs * log_probs).sum(dim=1)
            log_likelihood = -F.cross_entropy(logits_flat, targets_flat, reduction='none')

            # Reshape back to [B, D, H, W]
            shape = (B, D, H, W)
            preds = preds.view(shape).cpu()
            entropy = entropy.view(shape).cpu()
            log_likelihood = log_likelihood.view(shape).cpu()
            targets = grids.cpu()

            for i in range(B):
                hive_id = hive_idx[i].item()
                hive_results[hive_id]['ground_truth'].append(targets[i])
                hive_results[hive_id]['predictions'].append(preds[i])
                hive_results[hive_id]['entropy'].append(entropy[i])
                hive_results[hive_id]['log_likelihood'].append(log_likelihood[i])

    results = {'hives': []}
    for hive_id, data in hive_results.items():
        hive_dict = {
            'hive_id': hive_id,
            'ground_truth': torch.stack(data['ground_truth'], dim=0).numpy(),      # [T, D, H, W]
            'predictions': torch.stack(data['predictions'], dim=0).numpy(),
            'entropy': torch.stack(data['entropy'], dim=0).numpy(),
            'log_likelihood': torch.stack(data['log_likelihood'], dim=0).numpy(),
        }
        results['hives'].append(hive_dict)

    return results

import pytorch_lightning as pl
from pytorch_lightning import Trainer
from torch.utils.data import DataLoader
import torch
import os
import warnings

class SpatialClassifierModule(pl.LightningModule):
    def __init__(self, model, criterion, learning_rate, weight_decay, warmup_steps, cosine_steps):
        super(SpatialClassifierModule, self).__init__()
        self.model = model
        self.criterion = criterion
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.warmup_steps = warmup_steps
        self.cosine_steps = cosine_steps

    def forward(self, grids):
        return self.model(grids)

    def training_step(self, batch, batch_idx):
        grids, _, _ = batch
        grids = grids.to(self.device, dtype=torch.long)
        logits = self(grids)
        loss = self.criterion(logits, grids)
        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        grids, _, _ = batch
        grids = grids.to(self.device, dtype=torch.long)
        logits = self(grids)
        loss = self.criterion(logits, grids)
        self.log("val_loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def configure_optimizers(self):
        # Define parameter groups with separate learning rates
        param_groups = [
            {"params": self.model.embedding_weights, "lr": self.learning_rate * 10},  # Scale learning rate for embedding
        ]
    
        if hasattr(self.model, "positional_embedding"):
            param_groups.append(
                {"params": [self.model.positional_embedding], "lr": self.learning_rate * 10}
            )
    
        param_groups.append({
            "params": [p for n, p in self.model.named_parameters() 
                       if "embedding" not in n and "positional_embedding" not in n], 
            "lr": self.learning_rate
        })
    
        optimizer = torch.optim.AdamW(
            param_groups,
            weight_decay=self.weight_decay,
            betas=(0.9, 0.95),
        )
    
        scheduler = {
            "scheduler": torch.optim.lr_scheduler.SequentialLR(
                optimizer,
                schedulers=[
                    torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1/self.warmup_steps, total_iters=self.warmup_steps),
                    torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.cosine_steps)
                ],
                milestones=[self.warmup_steps]
            ),
            "interval": "step",
            "frequency": 1
        }
    
        return [optimizer], [scheduler]
    
def train_spatial_classifier(hive_grids, seed=42, **kwargs):
    pl.seed_everything(seed, workers=True)


    batch_size = kwargs.get('batch_size', 8)
    embedding_dim = kwargs.get('embedding_dim', 32)
    kernel_size = kwargs.get('kernel_size', 9)
    dilation = kwargs.get('dilation', 1)
    max_epochs = kwargs.get('max_epochs', 100)
    learning_rate = kwargs.get('learning_rate', 1e-3)
    weight_decay = kwargs.get('weight_decay', 1e-5)
    patience = kwargs.get('patience', 10)
    device = kwargs.get('device', 'cpu')

    # Warmup and cosine decay steps
    warmup_epochs = kwargs.get('warmup_epochs', 5)
    cosine_epochs = max_epochs - warmup_epochs

    dataset = GridDataset(hive_grids)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    
    num_classes = dataset.num_classes

    # Instantiate the model
    model = SpatialClassifier(num_classes=num_classes,
                              embedding_dim=embedding_dim,
                              kernel_size=kernel_size,
                              dilation=dilation,
                              depth=kwargs.get('depth', True),
                              height=kwargs.get('height', True),
                              width=kwargs.get('width', True),
                              pos_embed=kwargs.get('pos_embed', False),
                              context=kwargs.get('context', True),
                              interaction=kwargs.get('interaction', 0),
                              num_depth=kwargs.get('num_depth', dataset.shape[-3]),
                              num_height=kwargs.get('num_height', dataset.shape[-2]),
                              num_width=kwargs.get('num_width', dataset.shape[-1]),
                             ).to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device), reduction="mean")

    total_batches = len(dataloader)
    warmup_steps = warmup_epochs * total_batches
    cosine_steps = cosine_epochs * total_batches

    spatial_module = SpatialClassifierModule(
        model=model,
        criterion=criterion,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        warmup_steps=warmup_steps,
        cosine_steps=cosine_steps,
    )

    # Determine the accelerator and devices based on the device string
    if device.startswith("cuda"):
        # GPU: Extract device index from the string
        device_index = int(device.split(":")[1])
        accelerator = "gpu"
        devices = [device_index]  # Use specific GPU
    elif device == "cpu":
        # CPU: No GPU, fall back to CPU
        accelerator = "cpu"
        devices = None  # Lightning automatically uses the CPU
    else:
        raise ValueError(f"Unsupported device string: {device}")
    
    # Configure the Trainer
    trainer = Trainer(
        max_epochs=max_epochs,
        accelerator=accelerator,  # Dynamically set accelerator
        devices=devices,  # Use specific device or None for CPU
        log_every_n_steps=1,
        enable_progress_bar=True,
        enable_model_summary=False,
        logger=False,
        enable_checkpointing=False,
    )
    
    # Train the model
    trainer.fit(spatial_module, train_dataloaders=dataloader)
    
    return model

## 5. Configurations and the committed checkpoints

The eight model configurations (k=1 baseline and the seven 1D/2D/3D receptive
fields at kernel 3) map to the committed `spatial_classifier_v2_*` files. The
training call is kept commented out as the provenance reference.

In [10]:
import os
from itertools import product
import torch

# Define all configurations
configurations = list(product(
    [3],#[3, 5, 7, 9, 11, 13],                     # Kernel sizes
    product([True, False], repeat=3),        # depth, height, width
    [True],                           # pos_embed
    [True],                           # context
    [1],                                 # interaction: 0, 1, 2, 3, 4
))

dataset = GridDataset(hive_grids)
num_classes = dataset.num_classes

# Filter redundant configurations using `SpatialClassifier.get_kwargs`
unique_configs = []
for kernel_size, (depth, height, width), pos_embed, context, interaction in configurations:
    # Use `SpatialClassifier.get_kwargs` to validate and normalize configurations
    config = SpatialClassifier.get_kwargs(
        num_classes=num_classes,  # Provide required dummy values
        embedding_dim=128,
        kernel_size=kernel_size,
        dilation=1,
        depth=depth,
        height=height,
        width=width,
        pos_embed=pos_embed,
        context=context,
        interaction=interaction,  # Include the interaction parameter
        num_depth=dataset.shape[-3],
        num_height=dataset.shape[-2],
        num_width=dataset.shape[-1]
    )
    if config not in unique_configs:
        unique_configs.append(config)

# Define the corresponding filenames for unique configurations
trained_model_paths = [
    f'./models/spatial_classifier_v2_k{config["kernel_size"]}_d{config["depth"]}_h{config["height"]}_w{config["width"]}_pos{config["pos_embed"]}_ctx{config["context"]}_int{config["interaction"]}.pth'
    for config in unique_configs
]

# Print configurations and filenames
print("Configurations and filenames defined:")
for config, path in zip(unique_configs, trained_model_paths):
    print(f"Configuration: {config} => Path: {path}")

# Check for device availability
device = "cuda:0" if torch.cuda.is_available() else "cpu"

# Create a directory for saving models if it doesn't exist
os.makedirs('./models', exist_ok=True)


Configurations and filenames defined:
Configuration: {'num_classes': 6, 'embedding_dim': 128, 'kernel_size': 3, 'dilation': 1, 'depth': True, 'height': True, 'width': True, 'pos_embed': True, 'context': True, 'num_depth': 20, 'num_height': 52, 'num_width': 104, 'interaction': 1} => Path: ./models/spatial_classifier_v2_k3_dTrue_hTrue_wTrue_posTrue_ctxTrue_int1.pth
Configuration: {'num_classes': 6, 'embedding_dim': 128, 'kernel_size': 3, 'dilation': 1, 'depth': True, 'height': True, 'width': False, 'pos_embed': True, 'context': True, 'num_depth': 20, 'num_height': 52, 'num_width': 1, 'interaction': 1} => Path: ./models/spatial_classifier_v2_k3_dTrue_hTrue_wFalse_posTrue_ctxTrue_int1.pth
Configuration: {'num_classes': 6, 'embedding_dim': 128, 'kernel_size': 3, 'dilation': 1, 'depth': True, 'height': False, 'width': True, 'pos_embed': True, 'context': True, 'num_depth': 20, 'num_height': 1, 'num_width': 104, 'interaction': 1} => Path: ./models/spatial_classifier_v2_k3_dTrue_hFalse_wTrue_po

<cell 11>:219: UserWarning: `pos_embed` is set to True but `depth`, `height`, and `width` are all False. Setting `pos_embed` to False.
<cell 11>:226: UserWarning: `context` is set to True but `depth`, `height`, and `width` are all False. Setting `context` to False.
<cell 11>:242: UserWarning: `interaction` is set to 1 but `context` is False. Interaction requires `context` to be True. Setting `interaction` to 0.


In [11]:
# from tqdm.autonotebook import tqdm
# import torch

# # Filter every other config and model path
# unique_configs, trained_model_paths = unique_configs[::2], trained_model_paths[::2]
# # Uncomment the following line to use the other half of the configs and paths
# # unique_configs, trained_model_paths = unique_configs[1::2], trained_model_paths[1::2]

# # Train models based on predefined unique configurations
# with tqdm(total=len(unique_configs), desc="Training Spatial Classifiers", unit="model") as pbar:
#     for config, model_path in zip(unique_configs, trained_model_paths):

#         print(f"Training model with configuration: {config}...")

#         # Train the model using the unpacked config values
#         trained_model = train_spatial_classifier(
#             hive_grids=hive_grids,
#             batch_size=8,
#             embedding_dim=128,
#             kernel_size=config['kernel_size'],
#             max_epochs=200,
#             learning_rate=0.001,
#             weight_decay=1e-5,
#             warmup_epochs=5,
#             device=device,
#             depth=config['depth'],
#             height=config['height'],
#             width=config['width'],
#             pos_embed=config['pos_embed'],
#             context=config['context'],  # Pass the context flag
#             interaction=config['interaction'],
#         )

#         # Save the trained model
#         torch.save(trained_model.state_dict(), model_path)
#         print(f"Model saved to {model_path}.")

#         # Update the progress bar
#         pbar.update(1)

## 6. Evaluation set and model outputs

Build the 18-hive evaluation list (control, treated, simulated) and evaluate
each committed checkpoint, collecting per-hive likelihood outputs keyed by
configuration.

In [12]:
import numpy as np

def shuffle_frames_in_pairs_np(hive_array):
    """
    Shuffle frames in pairs (front and back) along the depth axis for a hive.

    Args:
        hive_array (np.ndarray): shape [time, depth, height, width]

    Returns:
        np.ndarray: same shape, shuffled in frame-pairs along depth
    """
    time, depth, height, width = hive_array.shape
    assert depth % 2 == 0, "Depth must be even for pair-wise frame shuffling."
    
    # Indices of frame pairs
    frame_pairs = [(i, i+1) for i in range(0, depth, 2)]
    shuffled_pairs = frame_pairs.copy()
    np.random.shuffle(shuffled_pairs)

    # Reconstruct the shuffled array
    shuffled_depth_slices = [hive_array[:, [i, j], :, :] for i, j in shuffled_pairs]
    shuffled_array = np.concatenate(shuffled_depth_slices, axis=1)

    return shuffled_array

# Split the list
control_hives = hive_grids[:6]
physically_shuffled_hives = hive_grids[6:]

# Create manually shuffled control hives
manually_shuffled_hives = [shuffle_frames_in_pairs_np(hive) for hive in control_hives]

# Concatenate all together
all_hives = control_hives + physically_shuffled_hives + manually_shuffled_hives


In [13]:
len(all_hives)

18

In [14]:
from tqdm.autonotebook import tqdm
import os
import torch
import numpy as np

results = {}
dataset = GridDataset(hive_grids)
num_classes = dataset.num_classes

# Evaluation loop with tqdm progress bar
for config, model_path in tqdm(zip(unique_configs, trained_model_paths), total=len(unique_configs), desc="Evaluating models"):

    if not os.path.exists(model_path):
        tqdm.write(f"Model file {model_path} does not exist. Skipping...")
        continue

    # Initialize model
    model = SpatialClassifier(
        num_classes=num_classes,
        embedding_dim=config['embedding_dim'],
        kernel_size=config['kernel_size'],
        depth=config['depth'],
        height=config['height'],
        width=config['width'],
        pos_embed=config['pos_embed'],
        context=config['context'],
        interaction=config['interaction'],
        num_depth=dataset.shape[-3],
        num_height=dataset.shape[-2],
        num_width=dataset.shape[-1],
    ).to(device)

    model.load_state_dict(torch.load(model_path, map_location=device))

    # Evaluate
    eval_output = evaluate_spatial_classifier(
        model=model,
        hive_grids=all_hives,
        device=device,
        batch_size=16
    )

    # Store the result using config as key
    config_key = (
        config['kernel_size'],
        config['depth'],
        config['height'],
        config['width'],
        config['pos_embed'],
        config['context'],
        config['interaction']
    )

    results[config_key] = eval_output  # contains 'hives': [ {hive_id, ground_truth, ...}, ... ]


Evaluating models: 100%|##########| 8/8 [00:19<00:00,  2.48s/it]


In [15]:
model_keys = list(results.keys())
model_keys

[(3, True, True, True, True, True, 1), (3, True, True, False, True, True, 1), (3, True, False, True, True, True, 1), (3, True, False, False, True, True, 1), (3, False, True, True, True, True, 1), (3, False, True, False, True, True, 1), (3, False, False, True, True, True, 1), (1, False, False, False, False, False, 0)]

## 7. Figures

Regenerate the paper's model outputs from the evaluation results:

* 3-D voxel view of ground-truth / predictions / likelihood
  (`3d_predictions_likelihood.png`, Figure 6B);
* likelihood over time for one configuration;
* **per-configuration supplemental figure**: mean likelihood for each
  individual 1D/2D/3D configuration, per treatment group
  (`supplement_config_likelihood.pdf`);
* **Figure 6**: 1D/2D/3D-averaged likelihood, per treatment group
  (`control_physical_simulated_likelihood.pdf`);
* receptive-field kernel legend (`spatial_legend_subset.pdf`).


In [16]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.colors as mcolors
import seaborn as sns
from tqdm import tqdm

def plot_model_scatter3d_simplified(results,
                                    model_keys,
                                    model_idx=0,
                                    hive_idx=0,
                                    week=0):
    """
    Plot simplified 3D metrics: ground truth, predictions, likelihood.
    Pairs of frames (front/back) are placed along width;
    frame pairs are stacked along depth.
    """

    simple_names = ['background', 'wood', 'empty_comb',
                    'any_brood', 'pollen', 'nectar_honey']
    num_classes = len(simple_names)
    class_palette = sns.color_palette("colorblind", num_classes)

    config_key = model_keys[model_idx]
    model_output = results[config_key]
    hive_data = model_output['hives'][hive_idx]

    preds = hive_data['predictions'][week]
    entropy = hive_data['entropy'][week]
    log_likelihood = hive_data['log_likelihood'][week]

    if 'ground_truth' in hive_data:
        ground_truth = hive_data['ground_truth'][week]
    else:
        for model_key in model_keys:
            gt_candidate = results[model_key]['hives'][hive_idx].get('ground_truth')
            if gt_candidate is not None:
                ground_truth = gt_candidate[week]
                break
        else:
            raise ValueError("Ground truth not found in any model.")

    likelihood = np.exp(log_likelihood)

    metrics = [
        ('ground_truth', ground_truth, 'class', 'Ground Truth Data'),
        ('predictions', preds, 'class', 'Model Predictions'),
        ('likelihood', likelihood, 'scalar', 'Model Likelihood'),
    ]

    fig = plt.figure(figsize=(10, 4), dpi=300)
    spacing = 50.0  # spacing between frame pairs along depth

    for i, (metric_key, data, mode, title) in enumerate(metrics):
        D, H, W = data.shape
        xs, ys, zs, cs, alphas = [], [], [], [], []

        print(f"Processing '{title}'...")
        for d in tqdm(range(D), desc=f"Collecting points for {metric_key}"):
            pair_idx = d // 2
            is_back = d % 2
            interleave_shift = -0.1 * W #0.33 * W
            side_shift = 0
            
            x_offset = is_back * (W - interleave_shift)
            y_offset = pair_idx * spacing + is_back * side_shift

            for y in range(H):
                for x in range(W):
                    gt_val = int(ground_truth[d, y, x])
                    if gt_val == 0:  # Always mask background
                        continue

                    val = data[d, y, x]

                    if mode == 'class':
                        val = int(val)
                        if val < 0 or val >= num_classes or val == 0:
                            continue
                        color = class_palette[val]
                        alpha = 1.0

                    elif metric_key == 'likelihood':
                        if gt_val == 1:  # Wood
                            color = class_palette[1]
                        else:
                            color = plt.cm.viridis(val)
                        alpha = 1.0
                        if alpha < 0.05:
                            continue

                    xs.append(x + x_offset)
                    ys.append(y_offset)
                    zs.append(H - y)
                    cs.append(color)
                    alphas.append(alpha)

        rgba_colors = [(*c[:3], a) for c, a in zip(cs, alphas)]

        ax = fig.add_subplot(1, 3, i + 1, projection='3d')
        sc = ax.scatter(xs, ys, zs, c=rgba_colors, s=0.33, edgecolors='none')
        
        ax.set_xlim(0, 2 * W + 1 - interleave_shift)
        ax.set_ylim(0, (D // 2) * spacing)
        ax.set_zlim(0, H)
        ax.set_box_aspect([2 * W + 1, (D // 2) * spacing, H])

        ax.set_xlabel("")
        ax.set_ylabel("")
        ax.set_zlabel("")
        ax.set_title(title, fontsize=10, pad=-100)

        ax.set_xticks([51.5 - 10, 51.5 * 3 - 10 - interleave_shift])
        ax.set_xticklabels(["Side A", "Side B"], fontsize=6)
        for label in ax.get_xticklabels():
            label.set_y(-10)  # negative moves them closer to axis

        ax.set_yticks([])
        ax.set_zticks([])

        # Clean background and panes
        ax.set_facecolor("white")
        for axis in [ax.xaxis, ax.yaxis, ax.zaxis]:
            # Hide pane background
            axis.set_pane_color((1, 1, 1, 0))
            
            # Hide grid lines
            if "grid" in axis._axinfo:
                axis._axinfo["grid"]['color'] = (1, 1, 1, 0)
            
            # Safely set edge color if pane info exists
            if "pane" in axis._axinfo:
                axis._axinfo["pane"]['edgecolor'] = 'black'
    fig.axes[0].set_ylabel("Depth (Back → Front)", fontsize=8)

    from matplotlib.cm import ScalarMappable
    from matplotlib.colors import ListedColormap, Normalize
    
    # === Colorbar for class labels ===
    class_labels = ['Wood', 'Empty\nComb', 'Brood', 'Pollen', 'Nectar/\nHoney']
    class_colors = class_palette[1:6]
    cmap_class = ListedColormap(class_colors)
    norm_class = Normalize(vmin=0, vmax=len(class_labels))
    
    sm_class = ScalarMappable(cmap=cmap_class, norm=norm_class)
    sm_class.set_array([])
    
    # === Colorbar for likelihood (viridis) ===
    cmap_likelihood = plt.cm.viridis
    norm_likelihood = Normalize(vmin=0, vmax=1)
    sm_likelihood = ScalarMappable(cmap=cmap_likelihood, norm=norm_likelihood)
    sm_likelihood.set_array([])
    
    # === Custom axes for both colorbars ===
    # [left, bottom, width, height] in figure fraction
    cbar_class_ax = fig.add_axes([0.25, 0.1, 0.2, 0.025])
    cbar_likelihood_ax = fig.add_axes([0.55, 0.1, 0.2, 0.025])
    
    # Plot the class colorbar
    cbar_class = plt.colorbar(
        sm_class,
        cax=cbar_class_ax,
        orientation='horizontal',
        ticks=np.arange(len(class_labels)) + 0.5,
    )
    cbar_class.ax.set_xticklabels(class_labels, fontsize=6)
    cbar_class.set_label("Class Labels", fontsize=8)
    
    # Plot the likelihood colorbar
    cbar_likelihood = plt.colorbar(
        sm_likelihood,
        cax=cbar_likelihood_ax,
        orientation='horizontal',
    )
    cbar_likelihood.ax.tick_params(labelsize=6)
    cbar_likelihood.set_label("Likelihood", fontsize=8)

    # plt.tight_layout()
    plt.subplots_adjust(top=0.8, hspace=0.0)
    plt.savefig('3d_predictions_likelihood.png', dpi=600, bbox_inches="tight")
    plt.show()


In [17]:
plot_model_scatter3d_simplified(
    results=results,
    model_keys=list(results.keys()),
    model_idx=0,
    hive_idx=1,
    week=8
)


Processing 'Ground Truth Data'...
Processing 'Model Predictions'...
Processing 'Model Likelihood'...


In [18]:
def plot_time_series_with_bootstrap_by_class(
    results,
    model_keys,
    class_names,
    model_idx=0,
    n_bootstrap=1000,
    ci=95
):
    import matplotlib.pyplot as plt
    import numpy as np
    import seaborn as sns
    from tqdm import tqdm

    config_key = model_keys[model_idx]
    model_output = results[config_key]
    hives = model_output['hives']

    num_classes = len(class_names)
    ignore_classes = []
    include_classes = [cls for cls in range(num_classes) if cls not in ignore_classes]
    class_colors = sns.color_palette("colorblind", num_classes)
    class_markers = ['o', 's', '^', 'D', 'v', '*']

    # Collect values per class and per week
    def collect_values(metric_func):
        values = {cls: {} for cls in include_classes}
        max_week = 0

        for hive in hives[:6]:
            entropy = hive['entropy']
            loglik = hive['log_likelihood']
            gt = hive.get('ground_truth')
            if gt is None:
                continue
            num_weeks = gt.shape[0]
            max_week = max(max_week, num_weeks)

            for t in range(num_weeks):
                for cls in include_classes:
                    mask = (gt[t] == cls)
                    if not np.any(mask):
                        continue
                    val = metric_func(entropy[t], loglik[t], mask)
                    values[cls].setdefault(t, []).append(val)
        return values, max_week

    # Bootstrap per class per week
    def bootstrap(values, max_week):
        means, lowers, uppers = {}, {}, {}
        for cls in include_classes:
            means[cls], lowers[cls], uppers[cls] = [], [], []
            for week in range(max_week):
                week_vals = values[cls].get(week, [])
                if not week_vals:
                    means[cls].append(np.nan)
                    lowers[cls].append(np.nan)
                    uppers[cls].append(np.nan)
                    continue
                samples = [
                    np.mean(np.random.choice(week_vals, size=len(week_vals), replace=True))
                    for _ in range(n_bootstrap)
                ]
                mu = np.mean(samples)
                lo = np.percentile(samples, (100 - ci) / 2)
                hi = np.percentile(samples, 100 - (100 - ci) / 2)
                means[cls].append(mu)
                lowers[cls].append(mu - lo)
                uppers[cls].append(hi - mu)
        return means, lowers, uppers

    # === Likelihood ===
    def likelihood(_, loglik_t, mask):
        return np.exp(loglik_t)[mask].mean()


    likelihood_vals, max_week = collect_values(likelihood)
    x_values = [w if w <= 10 else 10 + (w - 10) * 2 for w in range(max_week)]
    mean_l, low_l, high_l = bootstrap(likelihood_vals, max_week)

    fig, ax = plt.subplots(figsize=(6, 4), dpi=300)
    for cls in include_classes:
        y = mean_l[cls]
        yerr = [low_l[cls], high_l[cls]]
        ax.errorbar(
            x_values, y, yerr=yerr,
            fmt=class_markers[cls] + '-',
            label=class_names[cls],
            color=class_colors[cls],
            capsize=4
        )
    ax.set_title("Likelihood Over Time")
    ax.set_xlabel("Week")
    ax.set_ylabel("Likelihood")
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), borderaxespad=0.)
    plt.tight_layout()
    plt.show()


In [19]:
class_names = [
    "Background",
    "Wood",
    "Empty Comb",
    "Brood",
    "Pollen",
    "Nectar/Honey",
]

plot_time_series_with_bootstrap_by_class(
    results=results,
    model_keys=model_keys,
    class_names=class_names,
    model_idx=0,
    n_bootstrap=1000,
    ci=95
)


In [20]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import matplotlib.colors as mcolors
from mpl_toolkits.mplot3d import Axes3D
import copy


def plot_masked_kernel(ax, dims):
    kx, ky, kz = dims
    offset = [3 - (kx // 2), 3 - (ky // 2), 3 - (kz // 2)]

    for x in range(offset[0], offset[0] + kx):
        for y in range(offset[1], offset[1] + ky):
            for z in range(offset[2], offset[2] + kz):
                is_center = (x == offset[0] + kx // 2 and
                             y == offset[1] + ky // 2 and
                             z == offset[2] + kz // 2)

                if is_center:
                    facecolor = (0, 0, 0, 0.5)
                else:
                    facecolor = mcolors.to_rgba('gray', alpha=0.1)

                edgecolor = (0, 0, 0, 0.1)

                ax.bar3d(x - 0.5, y - 0.5, z - 0.5,
                         1, 1, 1,
                         color=facecolor,
                         edgecolor=edgecolor,
                         linewidth=0.25)

    ax.set_xlim(0, 7)
    ax.set_ylim(0, 7)
    ax.set_zlim(0, 7)
    ax.set_box_aspect([1, 1, 1])
    ax.axis("off")


def plot_likelihood_with_bootstrap(
    results, model_keys, num_classes, class_names,
    n_bootstrap=1000, ci=95, ax=None, title=""
):
    class_colors = sns.color_palette("colorblind", num_classes)
    class_markers = ['o', 's', '^', 'D', 'v', '*']

    spatial_order = [
        (True, False, False), (False, True, False), (False, False, True),
        (True, True, False), (True, False, True), (False, True, True),
        (True, True, True),
    ]
    x_labels = ["Depth", "Height", "Width", "D×H", "D×W", "H×W", "D×H×W"]
    kernel_shapes = [(1, 3, 1), (1, 1, 3), (3, 1, 1), (1, 3, 3), (3, 3, 1), (3, 1, 3), (3, 3, 3)]
    x = np.arange(len(spatial_order))

    # --- Baseline is pure chance: 1 / num_classes ---
    baseline_avg = 1.0 / num_classes

    def get_likelihood_matrix(hives):
        matrix = []
        for hive in hives:
            gt = hive['ground_truth']
            loglik = hive['log_likelihood']
            probs = np.exp(loglik)
            hive_vals = []
            for t in range(gt.shape[0]):
                gt_t = gt[t].flatten()
                prob_t = probs[t].reshape(-1)
                frame_vals = [prob_t[gt_t == cls].mean() if np.any(gt_t == cls) else np.nan
                              for cls in range(num_classes)]
                hive_vals.append(frame_vals)
            matrix.append(np.array(hive_vals))
        return matrix

    means = {cls: [] for cls in range(num_classes)}
    lowers = {cls: [] for cls in range(num_classes)}
    uppers = {cls: [] for cls in range(num_classes)}

    for dhw in spatial_order:
        key = next((k for k in model_keys if k[0] == 3 and k[1:4] == dhw), None)
        if not key:
            for cls in range(num_classes):
                means[cls].append(np.nan)
                lowers[cls].append(np.nan)
                uppers[cls].append(np.nan)
            continue

        hives = results[key]['hives']
        likelihood_matrix = get_likelihood_matrix(hives)
        arr = np.array(likelihood_matrix, dtype=object)

        for cls in range(num_classes):
            samples = []
            for _ in range(n_bootstrap):
                hive_ids = np.random.choice(len(arr), len(arr), replace=True)
                cls_vals = []
                for h in hive_ids:
                    weeks = arr[h]
                    week_ids = np.random.choice(len(weeks), len(weeks), replace=True)
                    cls_vals.extend([weeks[w][cls] for w in week_ids if not np.isnan(weeks[w][cls])])
                if cls_vals:
                    samples.append(np.mean(cls_vals))
            if samples:
                mu = np.mean(samples)
                lo = np.percentile(samples, (100 - ci) / 2)
                hi = np.percentile(samples, 100 - (100 - ci) / 2)
                means[cls].append(mu)
                lowers[cls].append(mu - lo)
                uppers[cls].append(hi - mu)
            else:
                means[cls].append(np.nan)
                lowers[cls].append(np.nan)
                uppers[cls].append(np.nan)

    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 4), dpi=300)
    else:
        fig = ax.figure

    for cls in range(num_classes):
        y = means[cls]
        yerr = [lowers[cls], uppers[cls]]
        ax.errorbar(
            x, y, yerr=yerr,
            fmt=class_markers[cls % len(class_markers)] + '-',
            label=class_names[cls],
            color=class_colors[cls % len(class_colors)],
            capsize=4
        )

    ax.axhline(baseline_avg, color='black', linestyle='dashed', linewidth=1.5,
               label=f"Baseline")
    ax.set_ylabel("Likelihood")
    ax.set_xticks(x)
    ax.set_xticklabels(x_labels, rotation=45, ha='right')
    ax.set_ylim(-0.05, 1.05)
    ax.set_title(title)
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)

    return fig, ax

def plot_control_physical_simulated(results, model_keys, num_classes, class_names, **kwargs):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), dpi=300, sharey=True)

    def run_on_slice(hive_slice, ax, title):
        sliced_results = copy.deepcopy(results)
        for key in sliced_results:
            sliced_results[key]['hives'] = sliced_results[key]['hives'][hive_slice]
        plot_likelihood_with_bootstrap(
            sliced_results, model_keys, num_classes, class_names,
            ax=ax, title=title, **kwargs
        )

    run_on_slice(slice(0, 6), axes[0], "Control (Unshuffled)")
    run_on_slice(slice(6, 12), axes[1], "Treatment (Shuffled)")
    run_on_slice(slice(12, 18), axes[2], "Simulated (Shuffled)")

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles, labels,
        loc='upper center', ncol=len(labels),
        fontsize=9, bbox_to_anchor=(0.5, 1.07)
    )
    axes[1].set_ylabel('')
    axes[2].set_ylabel('')
    plt.tight_layout()
    plt.savefig("control_physical_simulated_likelihood_full.pdf", bbox_inches="tight")
    plt.show()


In [21]:
model_keys = list(results.keys())
class_names = [
    "Background",
    "Wood",
    "Empty Comb",
    "Brood",
    "Pollen",
    "Nectar/Honey",
]

plot_control_physical_simulated(
    results=results,
    model_keys=model_keys,
    num_classes=6,
    class_names=class_names,
    n_bootstrap=1000,
    ci=95
)


In [22]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import matplotlib.colors as mcolors
from mpl_toolkits.mplot3d import Axes3D
import copy


def plot_masked_kernel(ax, dims):
    kx, ky, kz = dims
    offset = [3 - (kx // 2), 3 - (ky // 2), 3 - (kz // 2)]

    for x in range(offset[0], offset[0] + kx):
        for y in range(offset[1], offset[1] + ky):
            for z in range(offset[2], offset[2] + kz):
                is_center = (x == offset[0] + kx // 2 and
                             y == offset[1] + ky // 2 and
                             z == offset[2] + kz // 2)

                if is_center:
                    facecolor = (0, 0, 0, 0.5)
                else:
                    facecolor = mcolors.to_rgba('gray', alpha=0.1)

                edgecolor = (0, 0, 0, 0.1)

                ax.bar3d(x - 0.5, y - 0.5, z - 0.5,
                         1, 1, 1,
                         color=facecolor,
                         edgecolor=edgecolor,
                         linewidth=0.25)

    ax.set_xlim(0, 7)
    ax.set_ylim(0, 7)
    ax.set_zlim(0, 7)
    ax.set_box_aspect([1, 1, 1])
    ax.axis("off")


def plot_likelihood_with_bootstrap(
    results, model_keys, num_classes, class_names,
    n_bootstrap=1000, ci=95, ax=None, title=""
):
    import seaborn as sns
    from matplotlib import pyplot as plt
    import numpy as np

    class_colors = sns.color_palette("colorblind", num_classes)
    class_markers = ['o', 's', '^', 'D', 'v', '*']

    # Group by kernel dimensionality
    spatial_groups = {
        '1D': [
            (True, False, False),
            (False, True, False),
            (False, False, True),
        ],
        '2D': [
            (True, True, False),
            (True, False, True),
            (False, True, True),
        ],
        '3D': [
            (True, True, True),
        ]
    }
    x_labels = list(spatial_groups.keys())
    x = np.arange(len(x_labels))

    # Get baseline average likelihood
    baseline_key = next(k for k in model_keys if k[0] == 1 and k[1:4] == (False, False, False))
    baseline_hives = results[baseline_key]['hives'][:6]

    def get_likelihood_matrix(hives):
        matrix = []
        for hive in hives:
            gt = hive['ground_truth']
            loglik = hive['log_likelihood']
            probs = np.exp(loglik)
            hive_vals = []
            for t in range(gt.shape[0]):
                gt_t = gt[t].flatten()
                prob_t = probs[t].reshape(-1)
                frame_vals = [prob_t[gt_t == cls].mean() if np.any(gt_t == cls) else np.nan for cls in range(num_classes)]
                hive_vals.append(frame_vals)
            matrix.append(np.array(hive_vals))
        return matrix

    baseline_matrix = get_likelihood_matrix(baseline_hives)
    baseline_means = np.nanmean(np.concatenate(baseline_matrix, axis=0), axis=0)
    baseline_avg = np.nanmean(baseline_means)

    means = {cls: [] for cls in range(num_classes)}
    lowers = {cls: [] for cls in range(num_classes)}
    uppers = {cls: [] for cls in range(num_classes)}

    # Averaging over each spatial group
    for group_label in x_labels:
        dhw_list = spatial_groups[group_label]
        cls_samples = {cls: [] for cls in range(num_classes)}

        for dhw in dhw_list:
            key = next((k for k in model_keys if k[0] == 3 and k[1:4] == dhw), None)
            if not key:
                continue
            hives = results[key]['hives']
            matrix = get_likelihood_matrix(hives)
            arr = np.array(matrix, dtype=object)
            for cls in range(num_classes):
                for hive_weeks in arr:
                    for week in hive_weeks:
                        if not np.isnan(week[cls]):
                            cls_samples[cls].append(week[cls])

        for cls in range(num_classes):
            samples = []
            for _ in range(n_bootstrap):
                if not cls_samples[cls]:
                    samples.append(np.nan)
                    continue
                sample_vals = np.random.choice(cls_samples[cls], size=len(cls_samples[cls]), replace=True)
                samples.append(np.mean(sample_vals))

            samples = [s for s in samples if not np.isnan(s)]
            if samples:
                mu = np.mean(samples)
                lo = np.percentile(samples, (100 - ci) / 2)
                hi = np.percentile(samples, 100 - (100 - ci) / 2)
                means[cls].append(mu)
                lowers[cls].append(mu - lo)
                uppers[cls].append(hi - mu)
            else:
                means[cls].append(np.nan)
                lowers[cls].append(np.nan)
                uppers[cls].append(np.nan)

    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 4), dpi=300)
    else:
        fig = ax.figure

    for cls in range(num_classes):
        y = means[cls]
        yerr = [lowers[cls], uppers[cls]]
        ax.errorbar(
            x, y, yerr=yerr,
            fmt=class_markers[cls] + '-',
            label=class_names[cls],
            color=class_colors[cls],
            capsize=4
        )

    ax.axhline(baseline_avg, color='black', linestyle='dashed', linewidth=1.5, label="Baseline")
    ax.set_ylabel("Likelihood")
    ax.set_xticks(x)
    ax.set_xticklabels(x_labels, rotation=0)
    ax.set_ylim(-0.025, 1.025)
    ax.set_title(title, pad=10)
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)

    return fig, ax


def plot_control_physical_simulated(results, model_keys, num_classes, class_names, **kwargs):
    fig, axes = plt.subplots(1, 3, figsize=(10, 3.333), dpi=300, sharey=True)

    def run_on_slice(hive_slice, ax, title):
        sliced_results = copy.deepcopy(results)
        for key in sliced_results:
            sliced_results[key]['hives'] = sliced_results[key]['hives'][hive_slice]
        plot_likelihood_with_bootstrap(
            sliced_results, model_keys, num_classes, class_names,
            ax=ax, title=title, **kwargs
        )

    run_on_slice(slice(0, 6), axes[0], "Control (Unshuffled)")
    run_on_slice(slice(6, 12), axes[1], "Treatment (Shuffled)")
    run_on_slice(slice(12, 18), axes[2], "Simulated (Shuffled)")

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles, labels,
        loc='upper center', ncol=len(labels),
        fontsize=9, bbox_to_anchor=(0.5, 1.12)
    )
    axes[1].set_ylabel('')
    axes[2].set_ylabel('')
    for ax in axes:
        ax.set_xlim(-0.25, 2.25)
    plt.tight_layout()
    plt.savefig("control_physical_simulated_likelihood.pdf", bbox_inches="tight")
    plt.show()


In [23]:
model_keys = list(results.keys())
class_names = [
    "Background",
    "Wood",
    "Empty Comb",
    "Brood",
    "Pollen",
    "Nectar/Honey",
]

plot_control_physical_simulated(
    results=results,
    model_keys=model_keys,
    num_classes=6,
    class_names=class_names,
    n_bootstrap=1000,
    ci=95
)


In [24]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.colors as mcolors
import numpy as np

def plot_masked_kernel(ax, dims):
    kx, ky, kz = dims
    offset = [3 - (kx // 2), 3 - (ky // 2), 3 - (kz // 2)]

    for x in range(offset[0], offset[0] + kx):
        for y in range(offset[1], offset[1] + ky):
            for z in range(offset[2], offset[2] + kz):
                is_center = (x == offset[0] + kx // 2 and
                             y == offset[1] + ky // 2 and
                             z == offset[2] + kz // 2)
                facecolor = (0, 0, 0, 0.5) if is_center else mcolors.to_rgba('gray', alpha=0.1)
                edgecolor = (0, 0, 0, 0.1)

                ax.bar3d(x - 0.5, y - 0.5, z - 0.5,
                         1, 1, 1,
                         color=facecolor,
                         edgecolor=edgecolor,
                         linewidth=0.25)

    ax.set_xlim(0, 7)
    ax.set_ylim(0, 7)
    ax.set_zlim(0, 7)
    ax.set_box_aspect([1, 1, 1])
    ax.axis("off")

def plot_kernel_legend_subset():
    # Only the selected 3 shapes
    kernel_shapes = [(3, 1, 1), (3, 1, 3), (3, 3, 3)]
    x_labels = ["1D", "2D", "3D"]
    n = len(kernel_shapes)

    fig_width = 1 * n
    fig_height = 1.3
    fig = plt.figure(figsize=(fig_width, fig_height), dpi=300)

    left_margin = 0.005
    right_margin = 0.995
    spacing = (right_margin - left_margin) / n
    height = 0.8
    bottom = 0.12

    for i, (dims, label) in enumerate(zip(kernel_shapes, x_labels)):
        left = left_margin + i * spacing
        ax = fig.add_axes([left, bottom, spacing * 2.0, height], projection='3d')
        ax.patch.set_alpha(0.0)
        plot_masked_kernel(ax, dims)
        ax.view_init(elev=30, azim=120)
        ax.dist = 7
        ax.text(1, 6.8, 8.5, label, ha='center', va='bottom', fontsize=7.5)

    plt.savefig("spatial_legend_subset.pdf", bbox_inches="tight")
    plt.show()

# Call it
plot_kernel_legend_subset()
